# CXR-Triage: one-click training + evaluation

1. **Runtime → Change runtime type → T4 GPU** (free Colab GPU).
2. Fill in `PROJECT_ID` below to write AI reports with Gemini on Vertex AI, billed to your
   Google Cloud credit. Leave it empty to skip the AI part (the free template is used instead).
3. **Runtime → Run all.** Approve the Google Drive and Google sign-in pop-ups that appear in the
   first minute; after that it runs unattended (about 1–1.5 h).
4. Keep this tab open. Everything is saved to Google Drive (`MyDrive/cxr-triage`). If Colab
   disconnects, click Run all again: training resumes from the last checkpoint.

When finished, the GPU is released automatically.

In [1]:
PROJECT_ID = ""           # your Google Cloud project ID, e.g. "my-project-123"; "" skips AI reports
TRAIN_MINUTES = 100       # hard limit on training time
IMAGE_SIZE = 128          # 128 is fast; 224 is more accurate but about 3x slower
AUTO_SHUTDOWN = True      # release the GPU when everything is done

REPO = 'https://github.com/Sakshi1828/Healthcare-AI.git'
BRANCH = 'claude/keen-tesla-118bgd'

In [2]:
# All pop-ups happen here, at the start, so the rest can run unattended.
import os, sys, torch
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import auth, drive, userdata
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cxr-triage'
os.makedirs(f'{DRIVE}/data', exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)

if PROJECT_ID:
    auth.authenticate_user(project_id=PROJECT_ID)
    # Turn on the Vertex AI API for the project (no-op if it is already on).
    !gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}
    os.environ.update(GOOGLE_GENAI_USE_VERTEXAI='true', GOOGLE_CLOUD_PROJECT=PROJECT_ID,
                      GOOGLE_CLOUD_LOCATION='global')

# The repo is private unless you made it public: then add a GitHub token as the Colab secret
# GITHUB_TOKEN (key icon in the left sidebar).
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
CLONE_URL = REPO.replace('https://', f'https://{token}@') if token else REPO

GPU: Tesla T4
Mounted at /content/drive


In [3]:
%cd /content
!rm -rf healthcare-ai && git clone -q -b {BRANCH} {CLONE_URL} healthcare-ai
%cd /content/healthcare-ai
!pip install -q -e ".[serve,dev,llm]"
sys.path.insert(0, '/content/healthcare-ai/src')

/content
/content/healthcare-ai
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cxr-triage (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.0 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


In [4]:
# Check Vertex AI now (not after training) and pick two available Gemini models to compare.
GEMINI_MODELS = []
if PROJECT_ID:
    from google import genai
    client = genai.Client()

    def works(model):
        try:
            client.models.generate_content(model=model, contents='Reply with OK.')
            return True
        except Exception as err:
            print(f'  {model}: not available ({str(err)[:100]})')
            return False

    families = [
        ['gemini-3.5-flash', 'gemini-3-flash', 'gemini-3-flash-preview', 'gemini-2.5-flash'],
        ['gemini-3.5-flash-lite', 'gemini-3.1-flash-lite', 'gemini-3-flash-lite',
         'gemini-2.5-flash-lite'],
    ]
    for family in families:
        model = next((m for m in family if works(m)), None)
        if model:
            GEMINI_MODELS.append(model)
    if not GEMINI_MODELS:
        print('No Gemini model reachable. Check that the Vertex AI API is enabled in the project '
              'and billing is linked; training will continue and reports will use the template.')
print('Gemini models for the report evaluation:', GEMINI_MODELS or 'none (template only)')

Gemini models for the report evaluation: none (template only)


In [5]:
# Sanity check on synthetic data (~1 min): catches bugs before spending GPU time.
!pytest -q -x

.........................                                                [100%]
25 passed in 20.43s


In [6]:
# Dataset: NIH ChestX-ray14 via MedMNIST+, cached on Drive after the first download.
import shutil
from cxr_triage.config import load_config
from cxr_triage.data import load_split
os.makedirs('/content/data', exist_ok=True)
fname = f'chestmnist_{IMAGE_SIZE}.npz'
if os.path.exists(f'{DRIVE}/data/{fname}'):
    shutil.copy(f'{DRIVE}/data/{fname}', '/content/data/')
cfg = load_config('configs/default.yaml', ['data.root=/content/data', f'data.image_size={IMAGE_SIZE}'])
x, y = load_split(cfg, 'test')
print('test set:', x.shape)
if not os.path.exists(f'{DRIVE}/data/{fname}'):
    shutil.copy(f'/content/data/{fname}', f'{DRIVE}/data/')

100%|██████████| 1.43G/1.43G [10:08<00:00, 2.35MB/s]


test set: (22433, 128, 128)


In [ ]:
RUN = f'{DRIVE}/runs/densenet121_{IMAGE_SIZE}'
!python -m cxr_triage.train --config configs/default.yaml --resume run_dir={RUN} data.root=/content/data data.image_size={IMAGE_SIZE} train.max_minutes={TRAIN_MINUTES}

device=cuda amp=True run_dir=/content/drive/MyDrive/cxr-triage/runs/densenet121_128
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100% 30.8M/30.8M [00:00<00:00, 95.6MB/s]
{"epoch": 0, "train_loss": 0.70874961775098, "val_macro_auroc": 0.7761222131121065, "lr": 0.0003, "epoch_min": 2.512061623732249, "partial_epoch": false}
{"epoch": 1, "train_loss": 0.606985765945178, "val_macro_auroc": 0.8021925972603908, "lr": 0.0002939239460421746, "epoch_min": 2.464173456033071, "partial_epoch": false}
{"epoch": 2, "train_loss": 0.5777238625409242, "val_macro_auroc": 0.8153841109106679, "lr": 0.0002761880299246772, "epoch_min": 2.5241088112195333, "partial_epoch": false}


In [ ]:
!python -m cxr_triage.evaluate {RUN}

In [ ]:
# Grad-CAM gallery for the README.
import base64, io
import numpy as np, matplotlib.pyplot as plt
from PIL import Image as PILImage
from cxr_triage.inference import Predictor
pred = Predictor(f'{RUN}/bundle.pt', device='cuda')
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, name in zip(axes, ['cardiomegaly', 'effusion', 'pneumothorax', 'mass']):
    j = pred.labels.index(name)
    i = int(np.where(y[:, j] == 1)[0][0])
    out = pred.predict(PILImage.fromarray(x[i]), explain=True)
    ax.imshow(PILImage.open(io.BytesIO(base64.b64decode(out['gradcam']['png_base64']))))
    ax.set_title(f"GT: {name}\nCAM: {out['gradcam']['label']} | {out['triage']}")
    ax.axis('off')
fig.savefig(f'{RUN}/gradcam_gallery.png', dpi=150, bbox_inches='tight')

In [ ]:
# AI report evaluation on 100 held-out cases (template vs Gemini models), then pre-generate the
# demo's sample reports. Vertex AI usage here costs a few cents of your credit.
from cxr_triage.report_eval import main as report_eval
backends = ['template'] + [f'gemini:{m}' for m in GEMINI_MODELS]
report_eval([RUN, '--backends', ','.join(backends), '--n', '100', '--rpm', '60'])
report_eval([RUN, '--backends', backends[-1] if len(backends) == 1 else backends[1],
             '--bundle-samples'])

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{RUN}/report.md').read()))
display(Image(f'{RUN}/eval_plots.png'))
display(Image(f'{RUN}/gradcam_gallery.png'))
display(Markdown(open(f'{RUN}/report_eval.md').read().split('## Examples')[0]))
print('All results are in Google Drive:', RUN)

In [ ]:
# Release the GPU so no more time is used. Results are already on Drive.
if AUTO_SHUTDOWN:
    from google.colab import runtime
    runtime.unassign()